In [1]:
!pip install --upgrade transformers datasets jiwer evaluate Levenshtein huggingface_hub tensorboard tqdm 

  Using cached huggingface_hub-0.33.1-py3-none-any.whl.metadata (14 kB)
   ---------------------------------------- 0.0/515.4 kB ? eta -:--:--
    --------------------------------------- 10.2/515.4 kB ? eta -:--:--
    --------------------------------------- 10.2/515.4 kB ? eta -:--:--
   --- ----------------------------------- 41.0/515.4 kB 281.8 kB/s eta 0:00:02
   ------ -------------------------------- 92.2/515.4 kB 525.1 kB/s eta 0:00:01
   ----------------- ---------------------- 225.3/515.4 kB 1.1 MB/s eta 0:00:01
   ----------------------------------- ---- 460.8/515.4 kB 1.8 MB/s eta 0:00:01
   ---------------------------------------- 515.4/515.4 kB 1.9 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface-hub 0.33.0
    Uninstalling huggingface-hub-0.33.0:
      Successfully uninstalled huggingface-hub-0.33.0



[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from transformers import TrOCRProcessor, AutoTokenizer, VisionEncoderDecoderModel
import torch
from torch.utils.data import Dataset
from PIL import Image

from huggingface_hub import login
#access_token = user_secrets.get_secret("HF_Token")
login=login(token=access_token)

In [2]:
import torch
from torch.optim import AdamW
from transformers import (
    VisionEncoderDecoderModel,   # model class for image-to-text
    TrOCRProcessor,              # processor for TrOCR (image + tokenizer)
    AutoImageProcessor,          # for image preprocessing (EfficientNet)
    EfficientNetModel            # EfficientNet-B0 model
)

# Check for GPU (CUDA); this makes training much faster if available.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [3]:
import torch
import torch.nn as nn
from transformers import EfficientNetModel, VisionEncoderDecoderModel, TrOCRProcessor, BertTokenizer, AutoImageProcessor, AutoModelForCausalLM
from transformers.modeling_outputs import BaseModelOutput

# --- Define Model Sources ---
# Set paths to switch between local and Hugging Face models for the main components and EfficientNet encoder.
#MODEL_SOURCE_PATH = "./trocr-efficientnet-finetuned" 
MODEL_SOURCE_PATH = "ds1219/pandithv2"

# Path for the EfficientNet encoder itself.
# For Hugging Face model: "google/efficientnet-b0" (or other EfficientNet variants)
# For a locally saved EfficientNet: "./path/to/local/efficientnet-b0"
EFFICIENTNET_MODEL_NAME = "ds1219/pandithv2"
# EFFICIENTNET_MODEL_NAME = "./path/to/local/efficientnet-b0" # Example for local EfficientNet

# Define a path for saving the combined model
SAVE_MODEL_PATH = "./my_combined_trocr_efficientnet_model"

class SelfAttention(nn.Module):
    def __init__(self, embed_dim, num_heads=8):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        assert self.head_dim * num_heads == embed_dim, f"embed_dim ({embed_dim}) must be divisible by num_heads ({num_heads})"
        self.query = nn.Linear(embed_dim, embed_dim)
        self.key = nn.Linear(embed_dim, embed_dim)
        self.value = nn.Linear(embed_dim, embed_dim)
        self.scale = self.head_dim ** -0.5
        self.out_proj = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):
        batch_size, seq_len, _ = x.size()
        query = self.query(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        key = self.key(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        value = self.value(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        attention_scores = torch.matmul(query, key.transpose(-2, -1)) * self.scale
        attention_weights = torch.softmax(attention_scores, dim=-1)
        attended_values = torch.matmul(attention_weights, value)
        attended_values = attended_values.transpose(1, 2).contiguous().view(batch_size, seq_len, -1)
        output = self.out_proj(attended_values)
        return output

class EfficientNetCompatibleWrapper(nn.Module):
    def __init__(self, model_name="google/efficientnet-b0", target_hidden_size=768, num_attention_heads=8):
        super().__init__()
        self.model = EfficientNetModel.from_pretrained(model_name)
        
        # Determine efficientnet_hidden_size dynamically
        dummy_input_for_size_inference = torch.randn(1, 3, 224, 224) 
        with torch.no_grad():
            dummy_output = self.model(pixel_values=dummy_input_for_size_inference)
            self.efficientnet_hidden_size = dummy_output.last_hidden_state.shape[1]
        
        print(f"Dynamically determined EfficientNet output hidden size (before projection): {self.efficientnet_hidden_size}")
        self.target_hidden_size = target_hidden_size
        self.attention = SelfAttention(embed_dim=self.efficientnet_hidden_size, num_heads=num_attention_heads)
        self.projection = nn.Linear(self.efficientnet_hidden_size, self.target_hidden_size)
        
        # IMPORTANT: Set the config.hidden_size of the *encoder part* to match the target_hidden_size
        # This is what VisionEncoderDecoderModel checks against decoder.cross_attention_hidden_size
        self.model.config.hidden_size = target_hidden_size 
        
        # EfficientNet doesn't have cross-attention internally, so this should remain None
        self.model.config.cross_attention_hidden_size = None 
        
        self.config = self.model.config # The config property of the wrapper
        self.main_input_name = "pixel_values"

    def get_output_embeddings(self):
        return None
    
    def forward(
        self,
        pixel_values,
        output_attentions=None,
        output_hidden_states=None,
        return_dict=True,
        **kwargs
    ):
        efficientnet_forward_kwargs = {"pixel_values": pixel_values}
        if output_hidden_states is not None:
            efficientnet_forward_kwargs["output_hidden_states"] = output_hidden_states
        if return_dict is not None:
            efficientnet_forward_kwargs["return_dict"] = return_dict
        
        outputs = self.model(**efficientnet_forward_kwargs)
        
        last_hidden_state = outputs.last_hidden_state
        batch_size, num_channels, height, width = last_hidden_state.shape
        sequence_length = height * width
        
        reshaped_hidden_state = last_hidden_state.view(batch_size, num_channels, sequence_length).permute(0, 2, 1)
        attended_hidden_state = self.attention(reshaped_hidden_state)
        projected_hidden_state = self.projection(attended_hidden_state)
        
        if not return_dict:
            return (projected_hidden_state,)
        
        return BaseModelOutput(
            last_hidden_state=projected_hidden_state,
            hidden_states=outputs.hidden_states if output_hidden_states else None,
            attentions=None
        )

# --- End of Custom Wrapper Definition ---

## Create Model

In [34]:
# --- 1. Load Models and Processor ---
# Loads the tokenizer, image processor, and combines them into a TrOCRProcessor from the specified MODEL_SOURCE_PATH.
tokenizer = BertTokenizer.from_pretrained(MODEL_SOURCE_PATH)
image_processor = AutoImageProcessor.from_pretrained(MODEL_SOURCE_PATH)
processor = TrOCRProcessor(image_processor=image_processor, tokenizer=tokenizer)
print(f"TrOCRProcessor loaded successfully from: {MODEL_SOURCE_PATH}")


# --- MODIFIED: Load Decoder and Initialize VisionEncoderDecoderModel with Custom Encoder ---
# Load the entire pre-trained VisionEncoderDecoderModel from the checkpoint.
full_pretrained_model = VisionEncoderDecoderModel.from_pretrained(MODEL_SOURCE_PATH)

# Extract the decoder part and its hidden size from the loaded full model.
decoder_model = full_pretrained_model.decoder
original_decoder_hidden_size = decoder_model.config.hidden_size # This is the decoder's main hidden size

# --- CRITICAL FIX HERE ---
# Get the cross_attention_hidden_size from the decoder's config
# This is the size that the encoder's output *must* match.
decoder_cross_attention_hidden_size = getattr(decoder_model.config, 'cross_attention_hidden_size', None)

# If cross_attention_hidden_size is not explicitly set, it often defaults to the decoder's hidden_size
if decoder_cross_attention_hidden_size is None:
    decoder_cross_attention_hidden_size = original_decoder_hidden_size

print(f"Original decoder hidden size: {original_decoder_hidden_size}")
print(f"Decoder's expected cross_attention_hidden_size: {decoder_cross_attention_hidden_size}")


# Create EfficientNet Encoder with Compatible Wrapper, using the decoder's expected cross_attention_hidden_size
efficientnet_encoder = EfficientNetCompatibleWrapper(
    target_hidden_size=decoder_cross_attention_hidden_size, # Use the value the decoder expects!
    num_attention_heads=8 # You can adjust the number of attention heads here
)

# Initialize VisionEncoderDecoderModel by explicitly passing the custom EfficientNet encoder
# and the extracted, original decoder. This correctly constructs the combined model.
model = VisionEncoderDecoderModel(encoder=efficientnet_encoder, decoder=decoder_model)
print("VisionEncoderDecoderModel initialized with custom EfficientNet encoder and original decoder.")


# --- 5. Update Model Configuration (still necessary for complete config sync) ---
# Updates the main model's configuration with the new encoder's settings and ensures decoder cross-attention dimensions are correct.
model.config.encoder = efficientnet_encoder.config

# Ensure the decoder's config also reflects the correct encoder_hidden_size for cross-attention
if hasattr(model.config.decoder, 'encoder_hidden_size'):
    model.config.decoder.encoder_hidden_size = decoder_cross_attention_hidden_size
# If not present, add it. Some decoders might not explicitly have this, but it's good practice.
else:
    model.config.decoder.encoder_hidden_size = decoder_cross_attention_hidden_size


# Re-dimensions the decoder's cross-attention (k_proj and v_proj) layers to match the new EfficientNet encoder's output size.
# This loop iterates through the decoder's attention layers and resizes k_proj and v_proj to match
# the new encoder's output dimension.
for layer in model.decoder.model.decoder.layers:
    if hasattr(layer, 'encoder_attn'): 
        cross_attn = layer.encoder_attn
        
        # Make sure the input features to k_proj and v_proj are the encoder's output dimension
        if hasattr(cross_attn, 'k_proj'):
            old_k_proj = cross_attn.k_proj
            cross_attn.k_proj = nn.Linear(
                decoder_cross_attention_hidden_size,  # Input features will be from the encoder
                old_k_proj.out_features,        
                bias=old_k_proj.bias is not None
            )
        if hasattr(cross_attn, 'v_proj'):
            old_v_proj = cross_attn.v_proj
            cross_attn.v_proj = nn.Linear(
                decoder_cross_attention_hidden_size,  # Input features will be from the encoder
                old_v_proj.out_features,        
                bias=old_v_proj.bias is not None
            )
print("Decoder's cross-attention projection layers re-initialized to match encoder output.")


# --- 6. Set Special Tokens (CRITICAL FIX) ---
# Sets critical special token IDs (decoder_start_token_id, bos_token_id, eos_token_id, pad_token_id) for the model's generation process.
print("=== DEBUGGING TOKEN IDS ===")
print(f"Tokenizer type: {type(processor.tokenizer)}")
print(f"Available tokenizer attributes:")
for attr in ['pad_token_id', 'eos_token_id', 'bos_token_id', 'unk_token_id', 'cls_token_id', 'sep_token_id']:
    value = getattr(processor.tokenizer, attr, 'NOT_FOUND')
    print(f"  {attr}: {value}")

pad_token_id = processor.tokenizer.pad_token_id
eos_token_id = processor.tokenizer.eos_token_id
bos_token_id = getattr(processor.tokenizer, 'bos_token_id', None)

print(f"\nExtracted token IDs:")
print(f"  pad_token_id: {pad_token_id}")
print(f"  eos_token_id: {eos_token_id}")
print(f"  bos_token_id: {bos_token_id}")

decoder_start_token_id = (
    processor.tokenizer.cls_token_id 
    if hasattr(processor.tokenizer, 'cls_token_id') and processor.tokenizer.cls_token_id is not None 
    else (
        pad_token_id 
        if pad_token_id is not None 
        else (
            eos_token_id 
            if eos_token_id is not None 
            else 0
        )
    )
) if bos_token_id is None else bos_token_id
print(f"\nFinal decoder_start_token_id: {decoder_start_token_id}")

model.config.decoder_start_token_id = decoder_start_token_id
model.config.bos_token_id = decoder_start_token_id
model.config.eos_token_id = eos_token_id if eos_token_id is not None else decoder_start_token_id
model.config.pad_token_id = pad_token_id if pad_token_id is not None else decoder_start_token_id
model.config.vocab_size = model.config.decoder.vocab_size

# Also set these on the decoder's config directly for good measure
if hasattr(model.config, 'decoder'):
    model.config.decoder.decoder_start_token_id = decoder_start_token_id
    model.config.decoder.bos_token_id = decoder_start_token_id
    model.config.decoder.eos_token_id = model.config.eos_token_id
    model.config.decoder.pad_token_id = model.config.pad_token_id

print(f"\n=== FINAL TOKEN CONFIGURATION ===")
print(f"  decoder_start_token_id: {model.config.decoder_start_token_id}")
print(f"  bos_token_id: {model.config.bos_token_id}")
print(f"  eos_token_id: {model.config.eos_token_id}")
print(f"  pad_token_id: {model.config.pad_token_id}")

assert model.config.decoder_start_token_id is not None, "decoder_start_token_id is still None!"
print("✓ Token configuration successful!")

# --- 7. Move to Device ---
# Moves the model to the appropriate computational device (GPU if available, otherwise CPU).
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("EfficientNet encoder successfully integrated with parameter compatibility!")
print(f"Model moved to: {device}")
print(f"Encoder output hidden size (encoder.config.hidden_size): {model.config.encoder.hidden_size}")
print(f"Decoder expected cross_attention_hidden_size: {decoder_cross_attention_hidden_size}")
print(f"Encoder type: {type(model.encoder).__name__}")

# --- 8. Test the model to ensure it works ---
# Performs a quick forward pass and text generation test with dummy inputs to verify basic model functionality.
dummy_input = torch.randn(1, 3, 224, 224).to(device)
dummy_labels = torch.tensor([[1, 2, 3, model.config.eos_token_id]]).to(device)

print("\n=== TESTING MODEL FORWARD PASS ==ंभ")
with torch.no_grad():
    outputs = model(pixel_values=dummy_input, labels=dummy_labels)
    print(f"✓ Model forward pass successful! Loss: {outputs.loss.item():.4f}")
    print(f"✓ Output logits shape: {outputs.logits.shape}")

print("\n=== TESTING MODEL GENERATION ===")
with torch.no_grad():
    generated_ids = model.generate(pixel_values=dummy_input)
    print(f"✓ Model generation successful! Generated IDs shape: {generated_ids.shape}")
    dummy_pred_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    print(f"Dummy generated text: '{dummy_pred_text}'")

# --- 9. For Different EfficientNet Variants (Optional) ---
# Lists various EfficientNet variants and their corresponding output hidden sizes after projection.
# Note: These values are the *original* EfficientNet output channels, *before* your projection layer.
# They are for informational purposes if you were to bypass the projection,
# but your current setup projects them to the `decoder_cross_attention_hidden_size`.
efficientnet_original_output_channels = {
    "google/efficientnet-b0": 1280,
    "google/efficientnet-b1": 1280,
    "google/efficientnet-b2": 1408,
    "google/efficientnet-b3": 1536,
    "google/efficientnet-b4": 1792,
    "google/efficientnet-b5": 2048,
    "google/efficientnet-b6": 2304,
    "google/efficientnet-b7": 2560,
}

print(f"\n=== Original EfficientNet Output Channels (before your projection) ===")
for variant, hidden_size in efficientnet_original_output_channels.items():
    print(f"  {variant}: {hidden_size}")
print(f"\nYour encoder's actual output size (after projection) will be: {efficientnet_encoder.target_hidden_size}")


=== LOADING MODEL AND PROCESSOR FROM SAVED CHECKPOINT ===


Some weights of the model checkpoint at ds1219/pandithv2 were not used when initializing VisionEncoderDecoderModel: ['encoder.attention.key.bias', 'encoder.attention.key.weight', 'encoder.attention.out_proj.bias', 'encoder.attention.out_proj.weight', 'encoder.attention.query.bias', 'encoder.attention.query.weight', 'encoder.attention.value.bias', 'encoder.attention.value.weight', 'encoder.model.embeddings.batchnorm.bias', 'encoder.model.embeddings.batchnorm.num_batches_tracked', 'encoder.model.embeddings.batchnorm.running_mean', 'encoder.model.embeddings.batchnorm.running_var', 'encoder.model.embeddings.batchnorm.weight', 'encoder.model.embeddings.convolution.weight', 'encoder.model.encoder.blocks.0.depthwise_conv.depthwise_conv.weight', 'encoder.model.encoder.blocks.0.depthwise_conv.depthwise_norm.bias', 'encoder.model.encoder.blocks.0.depthwise_conv.depthwise_norm.num_batches_tracked', 'encoder.model.encoder.blocks.0.depthwise_conv.depthwise_norm.running_mean', 'encoder.model.encoder

Decoder loaded from: ds1219/pandithv2

--- Decoder Config Details ---
decoder.config.hidden_size: 1024
decoder.config.cross_attention_hidden_size: 768
----------------------------
Dynamically determined EfficientNet output hidden size (before projection): 1280
Custom EfficientNetCompatibleWrapper instantiated.
VisionEncoderDecoderModel re-initialized with custom encoder.
Model moved to device: cuda
Processor loaded from: ds1219/pandithv2
Set model.config.decoder_start_token_id to CLS token ID: 2
Set model.config.pad_token_id to: 0
Set model.config.eos_token_id to: None

--- VERIFYING LOADED COMPONENTS AFTER RE-INIT ---
Loaded model type: <class 'transformers.models.vision_encoder_decoder.modeling_vision_encoder_decoder.VisionEncoderDecoderModel'>
Loaded processor type: <class 'transformers.models.trocr.processing_trocr.TrOCRProcessor'>
Encoder type in model: <class '__main__.EfficientNetCompatibleWrapper'>
Is encoder instance of EfficientNetCompatibleWrapper? True
Final model.config.en

In [35]:
model.to(device)

VisionEncoderDecoderModel(
  (encoder): EfficientNetCompatibleWrapper(
    (model): EfficientNetModel(
      (embeddings): EfficientNetEmbeddings(
        (padding): ZeroPad2d((0, 1, 0, 1))
        (convolution): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=valid, bias=False)
        (batchnorm): BatchNorm2d(32, eps=0.001, momentum=0.99, affine=True, track_running_stats=True)
        (activation): SiLU()
      )
      (encoder): EfficientNetEncoder(
        (blocks): ModuleList(
          (0): EfficientNetBlock(
            (depthwise_conv): EfficientNetDepthwiseLayer(
              (depthwise_conv_pad): ZeroPad2d((0, 1, 0, 1))
              (depthwise_conv): EfficientNetDepthwiseConv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=same, groups=32, bias=False)
              (depthwise_norm): BatchNorm2d(32, eps=0.001, momentum=0.99, affine=True, track_running_stats=True)
              (depthwise_act): SiLU()
            )
            (squeeze_excite): EfficientNetSquee

# Load Model

In [5]:
# --- LOADING MODEL AND PROCESSOR ---
print("\n=== LOADING MODEL AND PROCESSOR FROM SAVED CHECKPOINT ===")

# Define device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Load the decoder from your fine-tuned checkpoint
decoder = VisionEncoderDecoderModel.from_pretrained("ds1219/pandithv2").decoder
print(f"Decoder loaded from: ds1219/pandithv2")

# Print decoder config to debug
print(f"\n--- Decoder Config Details ---")
print(f"decoder.config.hidden_size: {decoder.config.hidden_size}")
print(f"decoder.config.cross_attention_hidden_size: {decoder.config.cross_attention_hidden_size}")
print(f"----------------------------")

# 2. Instantiate your custom EfficientNet encoder
custom_encoder = EfficientNetCompatibleWrapper(
    model_name="google/efficientnet-b0", # Or whichever EfficientNet model you used as base
    target_hidden_size=decoder.config.cross_attention_hidden_size # The critical change from last time
)
print(f"Custom EfficientNetCompatibleWrapper instantiated.")

# 3. Create the VisionEncoderDecoderModel with your custom encoder and loaded decoder
model = VisionEncoderDecoderModel(encoder=custom_encoder, decoder=decoder)
print(f"VisionEncoderDecoderModel re-initialized with custom encoder.")

# Move model to device immediately after initialization
model.to(device)
print(f"Model moved to device: {device}")

# Load the processor
processor = TrOCRProcessor.from_pretrained("ds1219/pandithv2", use_fast=True)
print(f"Processor loaded from: ds1219/pandithv2")

# --- THE CORRECTED FIX FOR DECODER_START_TOKEN_ID ---
# TrOCR's tokenizer (RobertaTokenizer) typically uses `cls_token` (<s>) as the start token.
if processor.tokenizer.cls_token_id is None:
    raise ValueError(
        "CLS token ID not found in tokenizer. This is unexpected for TrOCR. "
        "Please ensure your tokenizer has a CLS token (e.g., '<s>')."
    )
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
print(f"Set model.config.decoder_start_token_id to CLS token ID: {model.config.decoder_start_token_id}")
# --- END CORRECTED FIX ---

# Also ensure pad_token_id and eos_token_id are set, as they are often needed for generation/training
if not hasattr(model.config, 'pad_token_id') or model.config.pad_token_id is None:
    model.config.pad_token_id = processor.tokenizer.pad_token_id
    print(f"Set model.config.pad_token_id to: {model.config.pad_token_id}")
if not hasattr(model.config, 'eos_token_id') or model.config.eos_token_id is None:
    model.config.eos_token_id = processor.tokenizer.eos_token_id
    print(f"Set model.config.eos_token_id to: {model.config.eos_token_id}")


# Verify loaded model and processor
print("\n--- VERIFYING LOADED COMPONENTS AFTER RE-INIT ---")
print(f"Loaded model type: {type(model)}")
print(f"Loaded processor type: {type(processor)}")
print(f"Encoder type in model: {type(model.encoder)}")
print(f"Is encoder instance of EfficientNetCompatibleWrapper? {isinstance(model.encoder, EfficientNetCompatibleWrapper)}")

# These config values should now match
print(f"Final model.config.encoder.hidden_size: {model.config.encoder.hidden_size}")
print(f"Final model.config.decoder.cross_attention_hidden_size: {model.config.decoder.cross_attention_hidden_size}")
print(f"Model config decoder_start_token_id: {model.config.decoder_start_token_id}")
print(f"Model config pad_token_id: {model.config.pad_token_id}")
print(f"Model config eos_token_id: {model.config.eos_token_id}")

print("\nModel and Processor loaded and verified. Ready for training/inference.")

# Remember to also move your input data (pixel_vals, labels) to the device in your training loop:
# pixel_vals = batch["pixel_values"].to(device)
# labels = batch["labels"].to(device)


=== LOADING MODEL AND PROCESSOR FROM SAVED CHECKPOINT ===


Some weights of the model checkpoint at ds1219/pandithv2 were not used when initializing VisionEncoderDecoderModel: ['encoder.attention.key.bias', 'encoder.attention.key.weight', 'encoder.attention.out_proj.bias', 'encoder.attention.out_proj.weight', 'encoder.attention.query.bias', 'encoder.attention.query.weight', 'encoder.attention.value.bias', 'encoder.attention.value.weight', 'encoder.model.embeddings.batchnorm.bias', 'encoder.model.embeddings.batchnorm.num_batches_tracked', 'encoder.model.embeddings.batchnorm.running_mean', 'encoder.model.embeddings.batchnorm.running_var', 'encoder.model.embeddings.batchnorm.weight', 'encoder.model.embeddings.convolution.weight', 'encoder.model.encoder.blocks.0.depthwise_conv.depthwise_conv.weight', 'encoder.model.encoder.blocks.0.depthwise_conv.depthwise_norm.bias', 'encoder.model.encoder.blocks.0.depthwise_conv.depthwise_norm.num_batches_tracked', 'encoder.model.encoder.blocks.0.depthwise_conv.depthwise_norm.running_mean', 'encoder.model.encoder

Decoder loaded from: ds1219/pandithv2

--- Decoder Config Details ---
decoder.config.hidden_size: 1024
decoder.config.cross_attention_hidden_size: 768
----------------------------
Dynamically determined EfficientNet output hidden size (before projection): 1280
Custom EfficientNetCompatibleWrapper instantiated.
VisionEncoderDecoderModel re-initialized with custom encoder.
Model moved to device: cuda
Processor loaded from: ds1219/pandithv2
Set model.config.decoder_start_token_id to CLS token ID: 2
Set model.config.pad_token_id to: 0
Set model.config.eos_token_id to: None

--- VERIFYING LOADED COMPONENTS AFTER RE-INIT ---
Loaded model type: <class 'transformers.models.vision_encoder_decoder.modeling_vision_encoder_decoder.VisionEncoderDecoderModel'>
Loaded processor type: <class 'transformers.models.trocr.processing_trocr.TrOCRProcessor'>
Encoder type in model: <class '__main__.EfficientNetCompatibleWrapper'>
Is encoder instance of EfficientNetCompatibleWrapper? True
Final model.config.en

In [6]:
model = torch.compile(model)
model.to(device)

OptimizedModule(
  (_orig_mod): VisionEncoderDecoderModel(
    (encoder): EfficientNetCompatibleWrapper(
      (model): EfficientNetModel(
        (embeddings): EfficientNetEmbeddings(
          (padding): ZeroPad2d((0, 1, 0, 1))
          (convolution): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=valid, bias=False)
          (batchnorm): BatchNorm2d(32, eps=0.001, momentum=0.99, affine=True, track_running_stats=True)
          (activation): SiLU()
        )
        (encoder): EfficientNetEncoder(
          (blocks): ModuleList(
            (0): EfficientNetBlock(
              (depthwise_conv): EfficientNetDepthwiseLayer(
                (depthwise_conv_pad): ZeroPad2d((0, 1, 0, 1))
                (depthwise_conv): EfficientNetDepthwiseConv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=same, groups=32, bias=False)
                (depthwise_norm): BatchNorm2d(32, eps=0.001, momentum=0.99, affine=True, track_running_stats=True)
                (depthwise_act): SiLU

# Load Data

In [9]:
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from PIL import Image

class OCRDatasetFromTSV(Dataset):
    """
    Custom OCR dataset that reads image paths and labels from a .tsv file.
    """
    def __init__(self, tsv_path, images_dir, image_processor, tokenizer, max_length=128):
        # Read the TSV file using pandas
        df = pd.read_csv(tsv_path, sep='\t').truncate(after=2000)
        self.image_paths = [f"{images_dir}/{fname}" for fname in df["file"].tolist()]
        self.texts = df["labels"].tolist()
        self.image_processor = image_processor
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        # Load and preprocess image
        image = Image.open(self.image_paths[idx]).convert("RGB")
        pixel_values = self.image_processor(images=image, return_tensors="pt").pixel_values.squeeze()

        # Tokenize text label
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        labels = encoding.input_ids.squeeze()
        labels[labels == self.tokenizer.pad_token_id] = -100

        return {"pixel_values": pixel_values, "labels": labels}

tsv_file_path = "C:\pandith\synthetic-handwritten\\train_labels.tsv"         # e.g., "train_data.tsv"
images_directory = "C:\pandith\synthetic-handwritten\\train"  # e.g., "./images"

# Image processor and tokenizer (as before)
image_processor = AutoImageProcessor.from_pretrained("google/efficientnet-b0")
tokenizer = processor.tokenizer  # from TrOCRProcessor

# Create dataset and dataloader
train_dataset = OCRDatasetFromTSV(tsv_file_path, images_directory, image_processor, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=8)

In [ ]:
from tqdm import tqdm  # for progress bars
import torch
from torch.optim import AdamW
from torch.utils.tensorboard import SummaryWriter # For TensorBoard logging
import os # To create logging directory

# Assuming 'model', 'train_loader', 'device' are already defined from previous steps
# And 'EFFICIENTNET_MODEL_NAME', 'MODEL_SOURCE_PATH', 'processor' are also defined

# --- Setup for Logging ---
# Define a directory for TensorBoard logs
log_dir = "./runs/efficientnet_trocr_training"
os.makedirs(log_dir, exist_ok=True)
writer = SummaryWriter(log_dir=log_dir)
print(f"TensorBoard logs will be saved to: {log_dir}")
# To view logs, run: tensorboard --logdir=./runs in your terminal

optimizer = AdamW(model.parameters(), lr=1e-5)  # Adam optimizer

num_epochs = 5
model.train()  # set model to training mode

global_step = 0 # Initialize a global step counter for TensorBoard

for epoch in range(num_epochs):
    epoch_loss = 0.0
    for batch_idx, batch in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}")):
        # Move input tensors to GPU
        pixel_vals = batch["pixel_values"].to(device)
        labels = batch["labels"].to(device)
        
        # Forward pass (compute loss)
        outputs = model(pixel_vals, labels=labels)
        loss = outputs.loss  # loss is a torch scalar
        
        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        current_loss_item = loss.item()
        epoch_loss += current_loss_item

        # --- Logging per-step metrics ---
        writer.add_scalar('Loss/step_train', current_loss_item, global_step)
        
        # You can add more frequent print statements if desired, e.g., every N steps
        if (batch_idx + 1) % 100 == 0: # Log every 100 steps for example
            tqdm.write(f"  Step {global_step} | Current Loss: {current_loss_item:.4f}")

        global_step += 1 # Increment global step after each batch

    avg_loss = epoch_loss / len(train_loader)
    print(f"Epoch {epoch+1} completed. Average Loss: {avg_loss:.4f}")

    # --- Logging epoch-level metrics ---
    writer.add_scalar('Loss/epoch_train_avg', avg_loss, epoch)

# Close the TensorBoard writer when training is done
writer.close()
print("Training complete. TensorBoard writer closed.")

TensorBoard logs will be saved to: ./runs/efficientnet_trocr_training


Epoch 1:   0%|          | 0/1251 [00:00<?, ?it/s]W0625 18:08:20.203000 7064 Lib\site-packages\torch\_inductor\utils.py:1250] [0/0_1] Not enough SMs to use max_autotune_gemm mode
`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.
Epoch 1:   8%|▊         | 100/1251 [06:27<18:21,  1.05it/s]  

  Step 99 | Current Loss: 6.7004


Epoch 1:  16%|█▌        | 200/1251 [08:00<14:13,  1.23it/s]

  Step 199 | Current Loss: 6.3364


...

  Step 1450 | Current Loss: 5.2614


Epoch 2:  24%|██▍       | 300/1251 [04:09<13:51,  1.14it/s]

  Step 1550 | Current Loss: 5.5488


Epoch 2:  29%|██▉       | 367/1251 [05:10<12:27,  1.18it/s]


KeyboardInterrupt: 

In [15]:
import os
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
from torch.utils.tensorboard import SummaryWriter

TSV_FILE_PATH = 'C:\pandith\synthetic-handwritten\\test_labels.tsv'
IMAGES_DIRECTORY_PATH = 'C:\pandith\synthetic-handwritten\\test'

train_dataset = OCRDatasetFromTSV(TSV_FILE_PATH, IMAGES_DIRECTORY_PATH, image_processor, tokenizer)
batch_size = 8
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)

print(f"Dataset loaded successfully with {len(train_dataset)} samples.")
print(f"DataLoader created with batch size {batch_size}.")

# Use a distinct log directory for evaluation to avoid mixing with training logs
log_dir = "./runs/ocr_evaluation"
os.makedirs(log_dir, exist_ok=True)
writer = SummaryWriter(log_dir=log_dir)
print(f"TensorBoard logs for evaluation will be saved to: {log_dir}")

print("\n=== EVALUATION AND METRICS CALCULATION ===")
model.eval() # Set model to evaluation mode

all_pred_texts = []
all_true_texts = []
total_eval_loss = 0.0 # Initialize variable to accumulate evaluation loss

OUTPUT_TSV_FILE_PATH = 'ground_truth_predictions.tsv'
print(f"Ground truth and predicted data will be saved to: {OUTPUT_TSV_FILE_PATH}")

with open(OUTPUT_TSV_FILE_PATH, 'w', encoding='utf-8') as f_out:
    f_out.write("Ground_Truth\tPredicted_Text\n")
    print(f"Processing {len(train_loader)} batches for metrics...")

    with torch.no_grad(): # Ensure no gradients are computed during evaluation
        for batch_idx, batch in enumerate(tqdm(train_loader, desc="Evaluating")):
            pixel_vals = batch["pixel_values"].to(device)
            labels = batch["labels"].to(device)

            # Compute loss during evaluation (similar to training forward pass)
            outputs = model(pixel_vals, labels=labels)
            loss = outputs.loss
            total_eval_loss += loss.item()

            generated_ids = model.generate(pixel_values=pixel_vals)

            labels_to_decode = labels.clone()
            pad_token_id = processor.tokenizer.pad_token_id
            if pad_token_id is None:
                # Fallback for models like GPT2 where pad_token_id is not set by default
                pad_token_id = processor.tokenizer.eos_token_id
                print(f"Warning: tokenizer.pad_token_id was None, setting it to tokenizer.eos_token_id ({pad_token_id})")

            labels_to_decode[labels_to_decode == -100] = pad_token_id if pad_token_id is not None else 0

            pred_texts = processor.batch_decode(generated_ids, skip_special_tokens=True)
            true_texts = processor.tokenizer.batch_decode(labels_to_decode, skip_special_tokens=True)

            all_pred_texts.extend(pred_texts)
            all_true_texts.extend(true_texts)

            for true_text, pred_text in zip(true_texts, pred_texts):
                f_out.write(f"{true_text}\t{pred_text}\n")

# Calculate average evaluation loss
avg_eval_loss = total_eval_loss / len(train_loader)
print(f"\nEvaluation completed. Average Evaluation Loss: {avg_eval_loss:.4f}")

# Log average evaluation loss to TensorBoard
# Using step 0 for a single evaluation run, or you could pass an epoch number if this is part of a larger loop
writer.add_scalar('Loss/eval_avg', avg_eval_loss, 0) # Log at step 0 for single evaluation summary

print(f"\nSuccessfully saved ground truth and predicted data to {OUTPUT_TSV_FILE_PATH}")
writer.close()
print("TensorBoard writer closed.")


Dataset loaded successfully with 2001 samples.
DataLoader created with batch size 8.
TensorBoard logs for evaluation will be saved to: ./runs/ocr_evaluation

=== EVALUATION AND METRICS CALCULATION ===
Ground truth and predicted data will be saved to: ground_truth_predictions.tsv
Processing 251 batches for metrics...


Evaluating: 100%|██████████| 251/251 [10:01<00:00,  2.40s/it] 


Evaluation completed. Average Evaluation Loss: 12.5790

Successfully saved ground truth and predicted data to ground_truth_predictions.tsv
TensorBoard writer closed.


In [16]:
from jiwer import cer, wer

# --- Calculate and Print Overall CER/WER ---
overall_cer = cer(all_true_texts, all_pred_texts) # Correct order for jiwer: (references, hypotheses)
overall_wer = wer(all_true_texts, all_pred_texts) # Correct order for jiwer: (references, hypotheses)

print(f"\n--- Overall Metrics Across All Samples ---")
print(f"Total samples evaluated: {len(all_pred_texts)}")
print(f"Overall Character Error Rate (CER): {overall_cer:.4f}")
print(f"Overall Word Error Rate (WER): {overall_wer:.4f}")

# --- Log metrics to TensorBoard ---
writer.add_scalar('Evaluation/CER', overall_cer, 0) # Use 0 as step since it's a final evaluation
writer.add_scalar('Evaluation/WER', overall_wer, 0) # Use 0 as step since it's a final evaluation
writer.close()
print("Metrics logged to TensorBoard. Writer closed.")


--- Overall Metrics Across All Samples ---
Total samples evaluated: 2001
Overall Character Error Rate (CER): 6.2076
Overall Word Error Rate (WER): 19.9810
Metrics logged to TensorBoard. Writer closed.


In [17]:
# Save the model
model.save_pretrained(SAVE_MODEL_PATH)
print(f"Model saved to: {SAVE_MODEL_PATH}")

# Save the processor (which includes tokenizer and image processor)
processor.save_pretrained(SAVE_MODEL_PATH)
print(f"Processor saved to: {SAVE_MODEL_PATH}")

Model saved to: ./my_combined_trocr_efficientnet_model
Processor saved to: ./my_combined_trocr_efficientnet_model


In [41]:
from huggingface_hub import HfApi

#api = HfApi()
repo_id = "ds1219/pandithv2" # Replace 'your_username' and repo name
#api.create_repo(repo_id=repo_id, private=True, exist_ok=True)
print(f"Repository {repo_id} created (or already exists).")

Repository ds1219/pandithv2 created (or already exists).


In [43]:
from huggingface_hub import upload_folder
import os

model_local_folder = "/kaggle/working/my_combined_trocr_efficientnet_model" # Your saved model directory

if not os.path.exists(model_local_folder):
    print(f"Error: Model directory not found at {model_local_folder}")
else:
    try:
        upload_folder(
            folder_path=model_local_folder,
            repo_id=repo_id,
            commit_message="Initial model and processor upload (using upload_folder)",
            repo_type="model",
            token=os.environ.get("HF_TOKEN") # Or pass your token directly
        )
        print(f"Model and processor files from {model_local_folder} uploaded successfully to {repo_id}!")
    except Exception as e:
        print(f"Failed to upload model and processor folder: {e}")

model.safetensors:   0%|          | 0.00/961M [00:00<?, ?B/s]

Model and processor files from /kaggle/working/my_combined_trocr_efficientnet_model uploaded successfully to ds1219/pandithv2!


In [9]:
print(tokenizer.encode("සිංහල"))

[2, 6253, 3]


In [12]:
print(print(processor.tokenizer.decode([7043])))

##වෙන
None
